In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

In [ ]:
colors = ["#5DADE2", "white", "#FF69B4"]  # Lighter sky blue and hot pink
custom_cmap = LinearSegmentedColormap.from_list("light_skyblue_to_hot_pink", colors)

## Load data

In [ ]:
adata_full = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

In [ ]:
tmp_subset = 'CD8_T_NK_ILC'

fig_dir = f"Reproducibility/Results/Plots/{tmp_subset}/"
os.makedirs(fig_dir, exist_ok = True)

adata = adata_full[adata_full.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1",
        "CD8_Tex_2","CD8_T_proliferative","NK_CD56_CD49a_Hi_CD103_Hi","NK_CD56_CD49a_Hi_CD103_Lo",
        "NK_CD56_CD49a_Lo","NK_CD56_dim",'ILC3','MAIT'], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)
celltype_colors = [
    '#1aafc9',  # ocean blue
    '#ff6f61',  # coral
    '#6a9f58',  # olive green
    '#f2b134',  # goldenrod
    '#d883b7',  # muted magenta
    '#6b4c9a',  # indigo (keep this one)
    '#c0592c',  # copper
    '#5ca4a9',  # teal gray
    '#3fb68b',  # jade green
    '#bc4b51',  # soft crimson
    '#7199c6',  # sky steel
    '#e84d8a',  # rose red
    '#da9429',  # warm ochre
    '#009e73',  # balanced green-cyan (replaces `#8e7dbe`)
]

In [ ]:
CD8_T_list = ["CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1","CD8_Tex_2","CD8_T_proliferative"]
adata_CD8 = adata[adata.obs['celltype'].isin(CD8_T_list)].copy()
adata_NK_ILC = adata[adata.obs['celltype'].isin(CD8_T_list)==False].copy()

## Visualization


### CD8<sup>+</sup> T/NK/ILC

UMAP

In [ ]:
# Fig.S8A
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    palette=celltype_colors,
    frameon=False,
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}FigureS8A.pdf", bbox_inches='tight')
plt.close()

Protein heatmap

In [ ]:
from muon import prot as pt
adt_df = adata.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins = ['CD8','NCAM','ckit', 'IL7Ra','CD161']

In [ ]:
# Fig.S8B
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}FigureS8B.pdf", bbox_inches='tight')
plt.close()

### CD8<sup>+</sup> T

RNA dotplot

In [ ]:
adata_RNA = adata_CD8[:, adata_CD8.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

adata_RNA.layers['scaled'] = sc.pp.scale(adata_RNA, copy=True).X

In [ ]:
marker_genes = ['CCR7',"TCF7","IL7R",'GZMK','CD28','EOMES',"KLRG1",'GZMH',
                "PRF1",'NKG7',"GNLY",'ITGAE','CCL4',
                "IFNG","TOX","CTLA4","HAVCR2","CXCL13","PDCD1","MKI67"
                ]

In [ ]:
# Fig.S8C
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby = "celltype",
              dendrogram=False,
              cmap=custom_cmap,
              dot_max = 0.8,
              layer = 'scaled',
              use_raw=False,
              vmin = -0.5,
              vmax = 0.5,
              show = False)
plt.savefig(f"{fig_dir}FigureS8C_RNA_dotplot.pdf", bbox_inches='tight')
plt.close()

Protein heatmap

In [ ]:
from muon import prot as pt
adt_df = adata_RNA.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_RNA.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata_RNA.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata_RNA.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata_RNA.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins = ['IL7Ra','CD55','CD57','KLRG1','ITGAE','PD1', 'CTLA4',"CD39",'CD71']

In [ ]:
# Fig.S8C
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}FigureS8C_protein_heatmap.pdf", bbox_inches='tight')
plt.close()

Gene program module scores

In [ ]:
adata_CD8_v2 = adata[adata.obs['celltype'].isin(["CD8_Tn","CD8_Tcm","CD8_Tem","CD8_Temra","CD8_Trm",
                                        "CD8_Tex_1","CD8_Tex_2"])].copy()

In [ ]:
tmp_path = 'Reproducibility/Results/VISIONR/UC_DOGMA_CD8_T_NK_ILC_signature_score_scenicplue_module.txt'
sig_df = pd.read_csv(tmp_path, index_col=0, sep='\t')
sig_df.columns = sig_df.columns + '_signature'

adata_CD8_v2.obs = pd.merge(left=adata_CD8_v2.obs, right=sig_df.loc[adata_CD8_v2.obs_names,:], 
                            how='left', left_index=True, right_index=True)

In [ ]:
# Fig.S8I
sc.set_figure_params(figsize=(2, 2)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata_CD8_v2,
    color=['GP1_signature','GP2_signature','GP3_signature','GP4_signature'],
    color_map = 'magma',    
    vmax="p95",
    vmin='p5',
    use_raw=False,
    frameon=False,
    ncols=1,
    wspace=0.1,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"{fig_dir}FigureS8I.pdf", bbox_inches='tight')
plt.close()

### NK/ILC

RNA dotplot

In [ ]:
adata_RNA = adata_NK_ILC[:, adata_NK_ILC.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

adata_RNA.layers['scaled'] = sc.pp.scale(adata_RNA, copy=True).X

In [ ]:
marker_genes = ["NCAM1",'KLRC1','KLRC2',
                'ITGAE','ITGA1',
                "PRF1","GZMB","ITGB2","CX3CR1",
                "IL7R","KIT","TCF7","CCR7","RORC",'KLRB1'         
                ]

In [ ]:
# Fig.S9A
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby = "celltype",
              dendrogram=False,
              cmap=custom_cmap,
              dot_max = 0.8,
              layer = 'scaled',
              use_raw=False,
              vmin = -0.5,
              vmax = 0.5,
              show = False)
plt.savefig(f"{fig_dir}FigureS9A_RNA_dotplot.pdf", bbox_inches='tight')
plt.close()

Protein heatmap

In [ ]:
from muon import prot as pt
adata_NK = adata_NK_ILC[adata_NK_ILC.obs['celltype'].isin(['MAIT','ILC3'])==False].copy()
adt_df = adata_NK.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_NK.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata_NK.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata_NK.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata_NK.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins = ["CD49a",'CD29','CD69','ITGAE','ITGB7','CD39','CD94','CD57','CD11b','CD16']

In [ ]:
# Fig.S9A
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}FigureS9A_protein_heatmap.pdf", bbox_inches='tight')
plt.close()